# 稠密检索完整流程演示：长文本分块 → 向量编码 → 检索

本 Notebook 完整演示稠密检索的全链路：
1. 原始长文本输入
2. 文本分块（Chunking）：手写实现 + RecursiveCharacterTextSplitter + MarkdownHeaderTextSplitter 三种方式对比
3. 加载 BGE 中文 Embedding 模型
4. 批量编码稠密向量
5. 构建 FAISS 向量索引
6. 执行稠密语义检索


In [5]:
# 安装依赖（首次运行执行一次即可）
%pip install sentence-transformers faiss-cpu langchain-text-splitters -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

# 设置 Hugging Face 国内镜像（需在导入 sentence_transformers 之前设置）
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from sentence_transformers import SentenceTransformer
import faiss
import re

W1019 15:45:34.564000 9561 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 1. 准备原始长文档
模拟真实场景中的《公司员工管理制度》长文本，未做分块处理。

In [2]:
# 原始长文档（未分块的完整文本）
raw_document = '''
第一章 总则
为规范公司日常管理，维护正常工作秩序，保障员工合法权益，根据国家相关法律法规，结合公司实际情况，制定本管理制度。本制度适用于与公司签订正式劳动合同的全体员工。公司倡导以人为本、高效协作的工作理念，鼓励员工与公司共同成长。

第二章 入职与试用期
新员工入职当日需提交身份证、学历证明、离职证明等相关材料，办理入职手续。劳动合同期限三个月以上不满一年的，试用期不得超过一个月；劳动合同期限一年以上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合同，试用期不得超过六个月。
试用期工资不得低于本单位相同岗位最低档工资或者劳动合同约定工资的百分之八十，并不得低于用人单位所在地的最低工资标准。试用期内员工若被证明不符合录用条件，公司可依法解除劳动合同。

第三章 工作时间与休息休假
公司实行标准工时制度，每日工作时间不超过八小时，平均每周工作时间不超过四十四小时。用人单位应当保证劳动者每周至少休息一日。
职工累计工作已满1年不满10年的，年休假5天；已满10年不满20年的，年休假10天；已满20年的，年休假15天。国家法定休假日、休息日不计入年休假的假期。
员工因私事需要本人处理的，可以申请事假。事假期间不支付工资。年度累计事假不得超过二十天，超过部分按旷工处理。

第四章 劳动报酬
用人单位应当按照劳动合同约定和国家规定，向劳动者及时足额支付劳动报酬。工资应当以货币形式按月支付给劳动者本人，不得克扣或者无故拖欠劳动者的工资。
公司于每月十五日发放上月工资，遇节假日则提前至最近的工作日发放。工资构成包括基本工资、岗位工资、绩效奖金、补贴等部分。个人所得税、社会保险个人缴纳部分由公司从员工工资中代扣代缴。

第五章 劳动合同的解除与终止
劳动者提前三十日以书面形式通知用人单位，可以解除劳动合同。劳动者在试用期内提前三日通知用人单位，可以解除劳动合同。
用人单位与劳动者协商一致，可以解除劳动合同。员工离职时需按照公司规定办理工作交接手续，交还公司财物、文件资料及办公设备。未办理交接手续擅自离职给公司造成损失的，公司有权要求赔偿。

第六章 工伤与社会保障
职工有下列情形之一的，应当认定为工伤：在工作时间和工作场所内，因工作原因受到事故伤害的；工作时间前后在工作场所内，从事与工作有关的预备性或者收尾性工作受到事故伤害的；在上下班途中，受到非本人主要责任的交通事故或者城市轨道交通、客运轮渡、火车事故伤害的。
公司依法为全体员工缴纳基本养老保险、基本医疗保险、工伤保险、失业保险、生育保险等社会保险。

第七章 附则
本制度自发布之日起施行，由人力资源部负责解释。公司有权根据经营发展需要对本制度进行修订并公示。
'''

print(f"原始文档总字符数：{len(raw_document)}")

原始文档总字符数：1132


## 2. 文本分块（Chunking）

下面演示三种分块方式：
- **方式一（手写实现）**：定长窗口分块 + 句子边界对齐 + 块间重叠
- **方式二（RecursiveCharacterTextSplitter）**：LangChain 官方封装的递归字符分块，工业界标准首选
- **方式三（MarkdownHeaderTextSplitter）**：按 Markdown 标题层级分块，结构化文档首选

### 2.1 方式一：手写分块函数

实现原则：
- 目标字符大小控制
- 句子边界对齐，避免截断完整语义
- 块间重叠（Overlap），避免边界信息丢失
- 窗口严格前进，保证分块过程一定会终止

In [3]:
def split_chunks(text, chunk_size=300, overlap=50):
    '''
    定长窗口分块 + 句子边界对齐
    chunk_size: 每个块的目标字符数
    overlap: 相邻块的重叠字符数
    '''
    # 参数校验：chunk_size 必须大于 overlap，保证窗口能正向推进
    if not (chunk_size > overlap >= 0):
        raise ValueError('要求 chunk_size > overlap >= 0')

    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        # 计算当前块的结束位置
        end = min(start + chunk_size, text_len)

        # 如果不是最后一块，尝试在句子边界（句号、换行）处切分
        if end < text_len:
            # 在窗口尾部 80 字符内向前找最近的句子结束符
            # low 下界保证 end - overlap > start，即 start 严格前进
            low = max(start + overlap + 1, end - 80)
            for i in range(end - 1, low - 1, -1):
                if text[i] in ['。', '\n', '；', '？']:
                    end = i + 1
                    break

        chunk = text[start:end].strip()
        if chunk:  # 跳过空块
            chunks.append(chunk)

        # 已处理到文末必须退出，否则会在尾部重叠区反复循环
        if end >= text_len:
            break

        # 移动起始位置，减去重叠部分
        start = end - overlap

    return chunks

# 执行分块
chunks_manual = split_chunks(raw_document, chunk_size=300, overlap=50)

print(f"【手写分块】共切分为 {len(chunks_manual)} 个块")
print("-" * 50)
for i, chunk in enumerate(chunks_manual[:2], 1):
    print(f"第{i}块（{len(chunk)}字符）：")
    print(chunk[:120] + "...")
    print("-" * 50)

【手写分块】共切分为 5 个块
--------------------------------------------------
第1块（252字符）：
第一章 总则
为规范公司日常管理，维护正常工作秩序，保障员工合法权益，根据国家相关法律法规，结合公司实际情况，制定本管理制度。本制度适用于与公司签订正式劳动合同的全体员工。公司倡导以人为本、高效协作的工作理念，鼓励员工与公司共同成长。

第...
--------------------------------------------------
第2块（293字符）：
上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合同，试用期不得超过六个月。
试用期工资不得低于本单位相同岗位最低档工资或者劳动合同约定工资的百分之八十，并不得低于用人单位所在地的最低工资标准。试用期内员工若被证明不符...
--------------------------------------------------


### 2.2 方式二：RecursiveCharacterTextSplitter（工业界标准首选）

LangChain 官方封装的递归字符分块器，核心优势：
- 递归尝试多级分隔符（段落→行→句号→逗号→空格），优先在语义边界切分
- 边界处理严谨，无死循环、空块等异常
- 支持自定义分隔符，适配中文、代码、表格等不同场景
- 轻量无依赖，只做分件，不捆绑其他 RAG 功能

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 初始化分块器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       # 单块目标字符数
    chunk_overlap=50,     # 块间重叠字符数
    separators=[           # 分隔符优先级，从高到低
        "\n\n",   # 段落分隔（优先）
        "\n",     # 行分隔
        "。",     # 中文句号
        "；",     # 分号
        "？", "！",
        "，",     # 逗号
        " "       # 空格
    ],
    keep_separator=True    # 保留句末标点等分隔符，避免中文标点丢失
)

# 执行分块，一行搞定
chunks_recursive = text_splitter.split_text(raw_document)

print(f"【RecursiveCharacterTextSplitter】共切分为 {len(chunks_recursive)} 个块")
print("-" * 50)
for i, chunk in enumerate(chunks_recursive[:2], 1):
    print(f"第{i}块（{len(chunk)}字符）：")
    print(chunk[:120] + "...")
    print("-" * 50)

【RecursiveCharacterTextSplitter】共切分为 6 个块
--------------------------------------------------
第1块（117字符）：
第一章 总则
为规范公司日常管理，维护正常工作秩序，保障员工合法权益，根据国家相关法律法规，结合公司实际情况，制定本管理制度。本制度适用于与公司签订正式劳动合同的全体员工。公司倡导以人为本、高效协作的工作理念，鼓励员工与公司共同成长。...
--------------------------------------------------
第2块（222字符）：
第二章 入职与试用期
新员工入职当日需提交身份证、学历证明、离职证明等相关材料，办理入职手续。劳动合同期限三个月以上不满一年的，试用期不得超过一个月；劳动合同期限一年以上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合...
--------------------------------------------------


### 2.3 方式三：MarkdownHeaderTextSplitter（结构化文档首选）

如果文档有清晰的标题结构（Markdown、带章节的制度文档、论文等），按标题层级分块效果远好于按字符硬切——它会天然保留语义单元，还能把标题作为元数据附加到分块上，大幅提升稠密检索的召回准确率。

下面准备一份 Markdown 格式的制度文档来演示。

In [5]:
# 准备 Markdown 格式的文档（带标题层级）
markdown_document = '''# 公司员工管理制度

## 第一章 总则
为规范公司日常管理，维护正常工作秩序，保障员工合法权益，根据国家相关法律法规，结合公司实际情况，制定本管理制度。本制度适用于与公司签订正式劳动合同的全体员工。

## 第二章 入职与试用期
新员工入职当日需提交身份证、学历证明、离职证明等相关材料，办理入职手续。

### 2.1 试用期期限
劳动合同期限三个月以上不满一年的，试用期不得超过一个月；劳动合同期限一年以上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合同，试用期不得超过六个月。

### 2.2 试用期工资
试用期工资不得低于本单位相同岗位最低档工资或者劳动合同约定工资的百分之八十，并不得低于用人单位所在地的最低工资标准。

## 第三章 工作时间与休息休假
公司实行标准工时制度，每日工作时间不超过八小时，平均每周工作时间不超过四十四小时。用人单位应当保证劳动者每周至少休息一日。

### 3.1 年休假
职工累计工作已满1年不满10年的，年休假5天；已满10年不满20年的，年休假10天；已满20年的，年休假15天。国家法定休假日、休息日不计入年休假的假期。

## 第四章 劳动报酬
用人单位应当按照劳动合同约定和国家规定，向劳动者及时足额支付劳动报酬。工资应当以货币形式按月支付给劳动者本人，不得克扣或者无故拖欠劳动者的工资。公司于每月十五日发放上月工资。

## 第五章 劳动合同的解除与终止
劳动者提前三十日以书面形式通知用人单位，可以解除劳动合同。劳动者在试用期内提前三日通知用人单位，可以解除劳动合同。员工离职时需按照公司规定办理工作交接手续。

## 第六章 工伤与社会保障
在上下班途中，受到非本人主要责任的交通事故伤害的，应当认定为工伤。公司依法为全体员工缴纳五险一金。
'''

In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 定义要识别的标题层级（Markdown # 数量 → 元数据字段名）
headers_to_split_on = [
    ("#", "一级标题"),
    ("##", "二级标题"),
    ("###", "三级标题"),
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 分块后每个块会自动携带所属的标题元数据
chunks_md = splitter.split_text(markdown_document)

print(f"【MarkdownHeaderTextSplitter】共切分为 {len(chunks_md)} 个块")
print("=" * 60)
for i, chunk in enumerate(chunks_md, 1):
    print(f"第{i}块：")
    print(f"  元数据：{chunk.metadata}")
    print(f"  内容：{chunk.page_content[:80]}...")
    print("-" * 60)

【MarkdownHeaderTextSplitter】共切分为 9 个块
第1块：
  元数据：{'一级标题': '公司员工管理制度', '二级标题': '第一章 总则'}
  内容：为规范公司日常管理，维护正常工作秩序，保障员工合法权益，根据国家相关法律法规，结合公司实际情况，制定本管理制度。本制度适用于与公司签订正式劳动合同的全体员工。...
------------------------------------------------------------
第2块：
  元数据：{'一级标题': '公司员工管理制度', '二级标题': '第二章 入职与试用期'}
  内容：新员工入职当日需提交身份证、学历证明、离职证明等相关材料，办理入职手续。...
------------------------------------------------------------
第3块：
  元数据：{'一级标题': '公司员工管理制度', '二级标题': '第二章 入职与试用期', '三级标题': '2.1 试用期期限'}
  内容：劳动合同期限三个月以上不满一年的，试用期不得超过一个月；劳动合同期限一年以上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合同，试用期不得...
------------------------------------------------------------
第4块：
  元数据：{'一级标题': '公司员工管理制度', '二级标题': '第二章 入职与试用期', '三级标题': '2.2 试用期工资'}
  内容：试用期工资不得低于本单位相同岗位最低档工资或者劳动合同约定工资的百分之八十，并不得低于用人单位所在地的最低工资标准。...
------------------------------------------------------------
第5块：
  元数据：{'一级标题': '公司员工管理制度', '二级标题': '第三章 工作时间与休息休假'}
  内容：公司实行标准工时制度，每日工作时间不超过八小时，平均每周工作时间不超过四十四小时。用人单位应当保证劳动者每周至少休息一日。...
------------------------------------------------

### 2.4 分块方案对比小结

| 方案 | 精度 | 速度 | 成本 | 适用场景 |
|------|------|------|------|----------|
| 手写分块 | 低 | 自定义 | 高 | 学习原理、极特殊定制需求 |
| RecursiveCharacterTextSplitter | 中 | 极快 | 极低 | 通用场景、大规模知识库、快速落地（工业界首选） |
| MarkdownHeaderTextSplitter | 高 | 快 | 低 | 结构化文档、Markdown、制度/手册/论文 |

> 工程经验：分块的核心不是「切得有多准」，而是「每个块包含一个完整语义点 + 带上下文元数据」。很多时候给字符分块加上标题、章节信息，效果提升比换更贵的分块算法还明显。

下面我们以 **RecursiveCharacterTextSplitter** 的输出作为后续检索的分块结果，继续完整流程演示。

## 3. 加载稠密检索模型
使用 BGE 中文 Embedding 模型（权重共享双塔结构）。

In [8]:
# 加载中文Embedding模型（权重共享双塔结构）
model = SentenceTransformer("BAAI/bge-base-zh-v1.5")
print(f"模型加载完成，向量维度：{model.get_embedding_dimension()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

模型加载完成，向量维度：768


## 4. 批量编码稠密向量（离线建库阶段）
将所有文本分块转换为固定维度的稠密向量。

In [9]:
# 使用 RecursiveCharacterTextSplitter 的结果进行编码
chunks = chunks_recursive

# 批量编码所有chunk为稠密向量
doc_embeddings = model.encode(chunks, normalize_embeddings=True)

print(f"向量矩阵形状：{doc_embeddings.shape}")
print(f"共 {doc_embeddings.shape[0]} 个向量，每个维度 {doc_embeddings.shape[1]}")

向量矩阵形状：(6, 768)
共 6 个向量，每个维度 768


In [10]:
print(doc_embeddings.shape)

(6, 768)


In [11]:
print(doc_embeddings)

[[ 1.44307902e-02 -6.85292184e-02 -1.94736719e-02 ... -1.10419458e-02
  -3.55742946e-02  2.35904362e-02]
 [ 3.38036916e-03 -1.01663142e-01 -1.84402820e-02 ... -2.28142925e-02
  -2.22467985e-02  5.75349759e-03]
 [ 3.57853808e-02 -8.08311924e-02 -7.25699429e-05 ... -4.74208109e-02
   4.87301545e-03  2.59821005e-02]
 [ 1.41668320e-02 -6.98733255e-02  1.47045660e-03 ... -4.34683897e-02
   1.25258404e-03  2.73620859e-02]
 [-1.23220123e-02 -6.87190965e-02 -2.81892102e-02 ... -4.55826260e-02
  -1.45230815e-02 -1.35067767e-02]
 [-3.42112258e-02 -5.62605634e-02  9.73095000e-03 ... -3.16932797e-02
  -7.34436465e-03  2.44113356e-02]]


## 5. 构建 FAISS 向量索引
使用内积索引（向量已归一化，等价于余弦相似度）。

索引类型
| 维度     | Flat       | IVF          | HNSW          | PQ/SQ          |
| -------- | ---------- | ------------ | ------------- | -------------- |
| 精度     | 精确       | 近似（可调） | 近似（可调）  | 近似，损失较大 |
| 是否训练 | 否         | 是           | Flat 版否     | 是             |
| 查询速度 | 慢（线性） | 快           | 很快          | 快且省内存     |
| 内存     | 原始向量   | 原始 / 压缩  | 原始 + 图结构 | 大幅压缩       |
| 建库成本 | 最低       | 中           | 高（建图慢）  | 中             |
| 适合规模 | 小到中     | 中到大       | 中到大        | 超大           |

经验阈值（非硬性，取决于维度和延迟要求）：$N < 10^5$ 用 Flat 完全够；$10^5 \sim 10^7$ 考虑 IVF-Flat 或 HNSW；$10^7$ 以上通常 IVF-PQ/SQ，必要时加 OPQ 和 GPU。

这里使用的是 Flat 索引，Flat 没有召回损失，结果可解释

In [12]:
dim = doc_embeddings.shape[1]
# 使用内积索引（向量已归一化，等价于余弦相似度）
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

print(f"索引构建完成，共 {index.ntotal} 条向量")

索引构建完成，共 6 条向量


## 6. 执行稠密语义检索
> 注意：BGE 模型需要为查询添加指令前缀，以区分 query/doc 角色，保证检索效果。

In [13]:
# 用户查询（BGE模型查询必须加指令前缀，区分query/doc角色）
query = "试用期工资怎么算？"
query_with_prompt = f"为这个句子生成表示以用于检索相关文章：{query}"

# 编码查询向量
query_embedding = model.encode([query_with_prompt], normalize_embeddings=True)

# 召回Top3最相关的分块
top_k = 3
scores, indices = index.search(query_embedding, top_k)

# 输出结果
print(f"用户查询：{query}\n")
print("稠密检索召回结果：")
print("=" * 60)
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    print(f"【第{rank}名 | 相似度：{score:.4f} | 块编号：{idx}】")
    print(chunks[idx])
    print("-" * 60)

用户查询：试用期工资怎么算？

稠密检索召回结果：
【第1名 | 相似度：0.5271 | 块编号：3】
第四章 劳动报酬
用人单位应当按照劳动合同约定和国家规定，向劳动者及时足额支付劳动报酬。工资应当以货币形式按月支付给劳动者本人，不得克扣或者无故拖欠劳动者的工资。
公司于每月十五日发放上月工资，遇节假日则提前至最近的工作日发放。工资构成包括基本工资、岗位工资、绩效奖金、补贴等部分。个人所得税、社会保险个人缴纳部分由公司从员工工资中代扣代缴。
------------------------------------------------------------
【第2名 | 相似度：0.5174 | 块编号：1】
第二章 入职与试用期
新员工入职当日需提交身份证、学历证明、离职证明等相关材料，办理入职手续。劳动合同期限三个月以上不满一年的，试用期不得超过一个月；劳动合同期限一年以上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合同，试用期不得超过六个月。
试用期工资不得低于本单位相同岗位最低档工资或者劳动合同约定工资的百分之八十，并不得低于用人单位所在地的最低工资标准。试用期内员工若被证明不符合录用条件，公司可依法解除劳动合同。
------------------------------------------------------------
【第3名 | 相似度：0.4672 | 块编号：2】
第三章 工作时间与休息休假
公司实行标准工时制度，每日工作时间不超过八小时，平均每周工作时间不超过四十四小时。用人单位应当保证劳动者每周至少休息一日。
职工累计工作已满1年不满10年的，年休假5天；已满10年不满20年的，年休假10天；已满20年的，年休假15天。国家法定休假日、休息日不计入年休假的假期。
员工因私事需要本人处理的，可以申请事假。事假期间不支付工资。年度累计事假不得超过二十天，超过部分按旷工处理。
------------------------------------------------------------


## 使用说明
1. 已内置 Hugging Face 国内镜像（hf-mirror.com），首次运行自动加速模型下载
2. 可修改 `chunk_size`、`chunk_overlap`（手写分块函数对应参数为 `overlap`）测试分块对召回效果的影响
3. 可修改查询语句，测试不同语义的检索效果
4. 如需切换其他分块方案，将后续编码步骤中的 `chunks_recursive` 替换为 `chunks_manual`；`chunks_md` 是 Document 对象列表，编码前需先转为文本：`chunks = [d.page_content for d in chunks_md]`